# This notebook will analyse if we could substituted HTF STR to LTF ITR

In [ ]:

import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "bigData"

PATH_1h = folder_path / "BTCUSDT-1h.json"
PATH_45m = folder_path / "BTCUSDT-45m.json"

# get 1H
with open(PATH_1h) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df_1h = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()
# end 1h

# get 45m
with open(PATH_45m) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df_45m = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()
# end 45m

# Halve memory: float64 -> float32 for price/volume columns
df_1h[["open", "high", "low", "close", "vol"]] = df_1h[["open", "high", "low", "close", "vol"]].astype("float32")
df_45m[["open", "high", "low", "close", "vol"]] = df_45m[["open", "high", "low", "close", "vol"]].astype("float32")

# beware  was truncated by the NotebookEdit tool on the previous write
print(f"1h Shape: {df_1h.shape} | Memory: {df_1h.memory_usage(deep=True).sum()/ 1e6:.2f} MB")
print(f"45m Shape: {df_45m.shape} | Memory: {df_45m.memory_usage(deep=True).sum()/ 1e6:.2f} MB")

1h Shape: (19981, 6) | Memory: 0.56 MB
45m Shape: (25641, 6) | Memory: 0.72 MB


# Verify STR count
- use it to represent 15m ITR

In [7]:
def _fix_break_of_structure(isXXX, isSource, high, low):
    """
    Ensure isXXX alternates strictly between 1 and -1.
    On a break (1,1 or -1,-1), insert the best counter swing between the two offending positions,
    but ONLY if it is structurally valid:
      - Inserting a  1 (high): candidate high must be > max(low[p1],  low[p2])
      - Inserting a -1 (low) : candidate low  must be < min(high[p1], high[p2])
    If no valid candidate exists, remove the less extreme of the two duplicates instead.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i - 1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 != v2:
                continue

            counter    = np.int8(-v1)
            candidates = np.arange(p1 + 1, p2)

            inserted = False
            if len(candidates) > 0:
                source_cands = candidates[isSource[candidates] == counter]
                pool = source_cands if len(source_cands) > 0 else candidates

                if counter == -1:
                    # Inserting a low: must dip below both surrounding highs
                    threshold  = min(high[p1], high[p2])
                    valid_pool = pool[low[pool] < threshold]
                else:
                    # Inserting a high: must rise above both surrounding lows
                    threshold  = max(low[p1], low[p2])
                    valid_pool = pool[high[pool] > threshold]

                if len(valid_pool) > 0:
                    best = (valid_pool[np.argmin(low[valid_pool])]
                            if counter == -1
                            else valid_pool[np.argmax(high[valid_pool])])
                    isXXX[best] = counter
                    inserted = True

            if not inserted:
                # No structurally valid candidate — drop the less extreme duplicate
                if v1 == 1:   # two highs: keep the higher one
                    isXXX[p1 if high[p1] < high[p2] else p2] = 0
                else:          # two lows: keep the lower one
                    isXXX[p1 if low[p1]  > low[p2] else p2] = 0

            changed = True
            break  # restart after each change

    return isXXX


In [8]:
def _fix_price_order(isXXX, high, low):
    """
    Fix price ordering violations by inserting 2 bridging marks instead of removing.
    For ITR/LTR: inserted marks need not be original swing points.
    Falls back to removal only if no valid bridge candidates exist.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i-1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 == 1 and v2 == -1 and low[p2] >= high[p1]:
                # Need a real low then a real high between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_lows = cands[low[cands] < high[p1]]
                if len(valid_lows) > 0:
                    best_low = valid_lows[np.argmin(low[valid_lows])]
                    after = np.arange(best_low + 1, p2)
                    valid_highs = after[high[after] > low[best_low]]
                    if len(valid_highs) > 0:
                        best_high = valid_highs[np.argmax(high[valid_highs])]
                        isXXX[best_low]  = -1
                        isXXX[best_high] = 1
                        changed = True
                        break
                # fallback: remove the bad low
                isXXX[p2] = 0
                changed = True
                break

            elif v1 == -1 and v2 == 1 and high[p2] <= low[p1]:
                # Need a real high then a real low between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_highs = cands[high[cands] > low[p1]]
                if len(valid_highs) > 0:
                    best_high = valid_highs[np.argmax(high[valid_highs])]
                    after = np.arange(best_high + 1, p2)
                    valid_lows = after[low[after] < high[best_high]]
                    if len(valid_lows) > 0:
                        best_low = valid_lows[np.argmin(low[valid_lows])]
                        isXXX[best_high] = 1
                        isXXX[best_low]  = -1
                        changed = True
                        break
                # fallback: remove the bad high
                isXXX[p2] = 0
                changed = True
                break

    return isXXX


In [9]:

# find str on window of 4
def strHunt(df, window=4):
    n = len(df)
    df = df.copy()

    high = df['high'].values
    low  = df['low'].values

    isSTR = np.zeros(n, dtype=np.int8)

    # --- First window: can't look backward, so determine direction from the span ---
    # if n > window + 1:
    #     diff = high[window + 1] - low[0]
    #     if diff > 0:
    #         isSTR[0] = -1   # start of bullish move → mark as low
    #     elif diff < 0:
    #         isSTR[0] = 1    # start of bearish move → mark as high

    # --- First window: method 2 just mark 0

    # --- Main loop: candles with a full symmetric window on both sides ---
    # Stop at n-window so the last `window` candles remain 0
    for i in range(window, n - window):
        win_high = high[i - window : i + window + 1]
        win_low  = low [i - window : i + window + 1]

        is_swing_high = high[i] >= win_high.max()   # local maximum
        is_swing_low  = low[i]  <= win_low.min()    # local minimum

        if is_swing_high and not is_swing_low:
            isSTR[i] = 1    # confirmed swing high
        elif is_swing_low and not is_swing_high:
            isSTR[i] = -1   # confirmed swing low
        # neither → stays 0
        # both, liquidity swept on both high and low → stays 0
        
    isSTR = _fix_break_of_structure(isSTR, isSTR, high, low)
    isSTR = _fix_price_order(isSTR, high, low)

    df['isSTR'] = isSTR
    return df

# execute -- 
df_1h = strHunt(df_1h)
df_45m = strHunt(df_45m)

In [10]:
# shift by 4, fill NaN with 0
# and rename the column to str_confirmed
STR_WINDOW = 4  # same window used in strHunt

# Shift forward by window → mark at bar t now only uses data up to bar t (fully causal)
df_1h['str_swing_confirmed']  = df_1h['isSTR'].shift(STR_WINDOW).fillna(0).astype('int8')
df_45m['str_swing_confirmed'] = df_45m['isSTR'].shift(STR_WINDOW).fillna(0).astype('int8')

# --- Count and compare against isITR target from journal ---
# Target: ~2,150 highs (+1) and ~2,150 lows (-1) per side
TARGET_PER_SIDE = 2_150

print(f"{'TF':<6} {'Highs (+1)':>12} {'Lows (-1)':>12} {'Total':>10}   vs target ~{TARGET_PER_SIDE*2:,}")
print("-" * 55)
for name, df in [('1H', df_1h), ('45m', df_45m)]:
    n_high = (df['str_swing_confirmed'] == 1).sum()
    n_low  = (df['str_swing_confirmed'] == -1).sum()
    total  = n_high + n_low
    diff   = total - TARGET_PER_SIDE * 2
    sign   = "+" if diff >= 0 else ""
    print(f"{name:<6} {n_high:>12,} {n_low:>12,} {total:>10,}   ({sign}{diff:,} vs target)")



TF       Highs (+1)    Lows (-1)      Total   vs target ~4,300
-------------------------------------------------------
1H            1,831        1,830      3,661   (-639 vs target)
45m           2,387        2,387      4,774   (+474 vs target)


In [12]:
def validate_markers(df, col):
    """
    Validate a marker column (e.g. 'isITR', 'isLTR') across the entire DataFrame.

    Checks two invariants:
      1. BOS (Break-of-Structure): consecutive non-zero entries must alternate sign (no 1,1 or -1,-1).
      2. Price order: after a high (1), the next low (-1) must be strictly below it,
         and after a low (-1), the next high (1) must be strictly above it.

    Returns
    -------
    dict with keys:
        'bos_violations'   : DataFrame of rows where sign repeats
        'order_violations' : DataFrame of rows where price order is broken
        'is_valid'         : True only if both checks pass
    """
    rows = df[df[col] != 0][['timestamp', 'high', 'low', col]].copy()
    rows['price_at_mark'] = rows.apply(
        lambda r: r['high'] if r[col] == 1 else r['low'], axis=1
    )

    vals = rows[col].values

    # --- BOS check: consecutive same-sign entries ---
    bos_mask = [False] + [vals[i] == vals[i - 1] for i in range(1, len(vals))]
    bos_violations = rows[bos_mask]

    # --- Price order check ---
    order_fail_idx = []
    for i in range(1, len(rows)):
        prev = rows.iloc[i - 1]
        curr = rows.iloc[i]
        if curr[col] == 1:          # current is a high → must be above prev low
            if curr['high'] <= prev['low']:
                order_fail_idx.append(rows.index[i])
        else:                        # current is a low → must be below prev high
            if curr['low'] >= prev['high']:
                order_fail_idx.append(rows.index[i])
    order_violations = rows.loc[order_fail_idx]

    # --- Print summary ---
    label = col
    total = len(rows)
    print(f"=== {label} validation ({total} marks across full dataset) ===")

    if len(bos_violations):
        print(f"  BOS violations : {len(bos_violations)}")
        print(bos_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  BOS violations : 0  — alternation OK")

    if len(order_violations):
        print(f"  Order violations: {len(order_violations)}")
        print(order_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  Order violations: 0  — price ordering OK")

    is_valid = len(bos_violations) == 0 and len(order_violations) == 0
    print(f"  Valid: {is_valid}\n")

    return {
        'bos_violations':   bos_violations,
        'order_violations': order_violations,
        'is_valid':         is_valid,
    }


# --- run validation on both columns ---
itr_1h_report = validate_markers(df_1h, 'isSTR')
itr_45m_report = validate_markers(df_45m, 'isSTR')

=== isSTR validation (3661 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True

=== isSTR validation (4774 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True

